In [29]:
from typing import Annotated, TypedDict, Literal
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage
from langchain_openrouter import ChatOpenRouter
from langgraph.checkpoint.memory import MemorySaver
from dotenv import load_dotenv

In [30]:
load_dotenv()

llm_model = ChatOpenRouter(model="meta-llama/llama-3-8b-instruct")

In [31]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [32]:
def chat_node(state: ChatState) -> ChatState:
    messages = state['messages']

    prompt = f'''Responses should be short/brief but concise and polite.'''

    input_prompt = [prompt] + messages

    response = llm_model.invoke(input_prompt).content

    return {'messages': [response]}

In [33]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)

In [34]:
thread_id = '1'

while True:
    input_message = input("Chat here (enter -1 to exit the chat) : ")

    if input_message == '-1':
        break

    config = {'configurable': {'thread_id': thread_id}}
    print("USER : ", input_message)
    input_state = {'messages': [HumanMessage(content=input_message)]}
    output_state = chatbot.invoke(input_state, config=config)
    print("AI : ", output_state['messages'][-1].content)

USER :  I'm Jason, how are you?
AI :  Hi Jason! I'm doing well, thanks for asking. How about you?
USER :  I'm doing good, would you tell me my name?
AI :  I'd be happy to! Your name is Jason.
USER :  How good are you at science?
AI :  I'm knowledgeable about a wide range of scientific topics, but I'm not a human scientist and my abilities have limitations. I can provide general information and help with answering specific questions, but I'm not a substitute for a professional expert.
USER :  How many planets does our solar system has?
AI :  Our solar system has 8 planets! (However, some sources may group Pluto as a planet, but most astronomers do not consider it a full-fledged planet.)
